# PyTorch 中的图卷积网络

NeurIPS'16 论文的 PyTorch 实现：
Convolutional Neural Networks on Graphs with Fast Localized Spectral Filtering
M Defferrard, X Bresson, P Vandergheynst
Advances in Neural Information Processing Systems, 3844-3852, 2016
[ArXiv 预印本](https://arxiv.org/abs/1606.09375)

改编自 Xavier Bresson 的仓库：[spectral_graph_convnets](https://github.com/xbresson/spectral_graph_convnets)，由 [Marc Lelarge](https://www.di.ens.fr/~lelarge/) 为 [dataflowr](https://dataflowr.github.io/website/) 改编

## 目标：

这份代码为 MNIST 分类任务提供了一个图卷积网络的简单示例。
图是 2D 网格的 8 近邻图。
图上的信号是向量化为 $28^2 \times 1$ 向量的 MNIST 图像。


In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import collections
import time
import numpy as np
import scipy
from functools import partial
import os

if torch.cuda.is_available():
    print('cuda available')
    dtypeFloat = torch.cuda.FloatTensor
    dtypeLong = torch.cuda.LongTensor
    torch.cuda.manual_seed(1)
else:
    print('cuda not available')
    dtypeFloat = torch.FloatTensor
    dtypeLong = torch.LongTensor
    torch.manual_seed(1)

## 下载数据

如果你在 colab 上运行，按照下面的说明操作。

如果你克隆了仓库，直接跳到临时 hack 部分。


### Colab 设置

如果你在 colab 上运行这个 notebook，请取消下面这些单元格的注释（并运行）。


In [ ]:
#!wget www.di.ens.fr/~lelarge/graphs.tar.gz

In [ ]:
#!tar -zxvf graphs.tar.gz

In [ ]:
#%cd graphs

### 临时 hack

如果在 colab 上运行（或者你已经有 MNIST），就不需要，见这个[问题](https://github.com/pytorch/vision/issues/3497)


## 加载数据


In [ ]:
def check_mnist_dataset_exists(path_data='./'):
    flag_train_data = os.path.isfile(path_data + 'mnist/train_data.pt') 
    flag_train_label = os.path.isfile(path_data + 'mnist/train_label.pt') 
    flag_test_data = os.path.isfile(path_data + 'mnist/test_data.pt') 
    flag_test_label = os.path.isfile(path_data + 'mnist/test_label.pt') 
    if flag_train_data==False or flag_train_label==False or flag_test_data==False or flag_test_label==False:
        print('MNIST dataset preprocessing...')
        import torchvision
        import torchvision.transforms as transforms
        trainset = torchvision.datasets.MNIST(root=path_data + 'mnist/temp', train=True,
                                                download=True, transform=transforms.ToTensor())
        testset = torchvision.datasets.MNIST(root=path_data + 'mnist/temp', train=False,
                                               download=True, transform=transforms.ToTensor())
        train_data=torch.Tensor(60000,28,28)
        train_label=torch.LongTensor(60000)
        for idx , example in enumerate(trainset):
            train_data[idx]=example[0].squeeze()
            train_label[idx]=example[1]
        torch.save(train_data,path_data + 'mnist/train_data.pt')
        torch.save(train_label,path_data + 'mnist/train_label.pt')
        test_data=torch.Tensor(10000,28,28)
        test_label=torch.LongTensor(10000)
        for idx , example in enumerate(testset):
            test_data[idx]=example[0].squeeze()
            test_label[idx]=example[1]
        torch.save(test_data,path_data + 'mnist/test_data.pt')
        torch.save(test_label,path_data + 'mnist/test_label.pt')
    return path_data


_ = check_mnist_dataset_exists()

In [ ]:
#如果你想用小的数据集（cpu 上），取消注释。
#nb_selected_train_data = 500
#nb_selected_test_data = 100

train_data=torch.load('mnist/train_data.pt').reshape(60000,784).numpy()
#train_data = train_data[:nb_selected_train_data,:]
print(train_data.shape)

train_labels=torch.load('mnist/train_label.pt').numpy()
#train_labels = train_labels[:nb_selected_train_data]
print(train_labels.shape)

test_data=torch.load('mnist/test_data.pt').reshape(10000,784).numpy()
#test_data = test_data[:nb_selected_test_data,:]
print(test_data.shape)

test_labels=torch.load('mnist/test_label.pt').numpy()
#test_labels = test_labels[:nb_selected_test_data]
print(test_labels.shape)

In [ ]:
from lib.grid_graph import grid_graph
from lib.coarsening import coarsen, HEM, compute_perm, perm_adjacency
from lib.coarsening import perm_data

# 构造图
t_start = time.time()
grid_side = 28
number_edges = 8
metric = 'euclidean'


######## ####### 在这里填你的图邻接矩阵 ########
A = grid_graph(grid_side,number_edges,metric)  # 创建欧氏网格的图
######## ####### 在这里填你的图邻接矩阵 ########

In [ ]:
def laplacian(W, normalized=True):
    """Return graph Laplacian"""
    I = scipy.sparse.identity(W.shape[0], dtype=W.dtype)

    #W += I
    # 度数矩阵。
    d = W.sum(axis=0)

    # 拉普拉斯矩阵。
    if not normalized:
        D = scipy.sparse.diags(d.A.squeeze(), 0)
        L = D - W
    else:
        #
        #
        # 在这里写归一化拉普拉斯矩阵的代码
        #
        #
        pass

    assert np.abs(L - L.T).mean() < 1e-8
    assert type(L) is scipy.sparse.csr.csr_matrix
    return L

In [ ]:
def rescale_L(L, lmax=2):
    """Rescale Laplacian eigenvalues to [-1,1]"""
    M, M = L.shape
    I = scipy.sparse.identity(M, format='csr', dtype=L.dtype)
    L /= lmax * 2
    L -= I
    return L 

def lmax_L(L):
    """Compute largest Laplacian eigenvalue"""
    return scipy.sparse.linalg.eigsh(L, k=1, which='LM', return_eigenvectors=False)[0]

In [ ]:
# 计算粗化后的图
coarsening_levels = 4

L, perm = coarsen(A, coarsening_levels, partial(laplacian, normalized=False))

# 计算图拉普拉斯矩阵的最大特征值
lmax = []
for i in range(coarsening_levels):
    lmax.append(lmax_L(L[i]))
print('lmax: ' + str([lmax[i] for i in range(coarsening_levels)]))

# 重新索引节点以满足二叉树结构
train_data = perm_data(train_data, perm)
test_data = perm_data(test_data, perm)

print('Execution time: {:.2f}s'.format(time.time() - t_start))
del perm

这里，我们实现了池化层，并计算了列表 `L`，它包含每一层图的拉普拉斯矩阵。

<font color='red'>问题：各个池化的尺寸是多少？</font>


# 图卷积网络 LeNet5

## 层：CL32-MP4-CL64-MP4-FC512-FC10

如上所述，这个网络有 2 个图卷积层和 2 个大小为 4 的池化层。

<font color='red'>问题：图卷积层会取列表 `L` 中的哪些图？</font>

在下面的代码里，你需要补全 `graph_conv_cheby` 和 `graph_max_pool`。

提示：每次对维度做 permute 之后，加上 `contiguous` 是安全的，就像下面这样：
`x0 = x.permute(1,2,0).contiguous()` 见[这里](https://discuss.pytorch.org/t/call-contiguous-after-every-permute-call/13190/2)


In [ ]:
class Graph_ConvNet_LeNet5(nn.Module):
    
    def __init__(self, net_parameters):
        
        print('Graph ConvNet: LeNet5')
        
        super(Graph_ConvNet_LeNet5, self).__init__()
        
        # 参数
        D, CL1_F, CL1_K, CL2_F, CL2_K, FC1_F, FC2_F = net_parameters
        FC1Fin = CL2_F*(D//16)
        
        # 图 CL1
        self.cl1 = nn.Linear(CL1_K, CL1_F) 
        self.init_layers(self.cl1, CL1_K, CL1_F)
        self.CL1_K = CL1_K; self.CL1_F = CL1_F; 
        
        # 图 CL2
        self.cl2 = nn.Linear(CL2_K*CL1_F, CL2_F) 
        self.init_layers(self.cl2, CL2_K*CL1_F, CL2_F)
        self.CL2_K = CL2_K; self.CL2_F = CL2_F; 

        # FC1
        self.fc1 = nn.Linear(FC1Fin, FC1_F) 
        self.init_layers(self.fc1, FC1Fin, FC1_F)
        self.FC1Fin = FC1Fin
        
        # FC2
        self.fc2 = nn.Linear(FC1_F, FC2_F)
        self.init_layers(self.fc2, FC1_F, FC2_F)

        # 参数数量
        nb_param = CL1_K* CL1_F + CL1_F  # CL1
        nb_param += CL2_K* CL1_F* CL2_F + CL2_F  # CL2
        nb_param += FC1Fin* FC1_F + FC1_F  # FC1
        nb_param += FC1_F* FC2_F + FC2_F  # FC2
        print('nb of parameters=',nb_param,'\n')
        
        
    def init_layers(self, W, Fin, Fout):

        scale = np.sqrt( 2.0/ (Fin+Fout) )
        W.weight.data.uniform_(-scale, scale)
        W.bias.data.fill_(0.0)

        return W
        
        
    def graph_conv_cheby(self, x, cl, L, lmax, Fout, K):
        # 参数
        # B = 批大小
        # V = 顶点数
        # Fin = 输入特征数
        # Fout = 输出特征数
        # K = Chebyshev 阶数 & 支撑大小
        B, V, Fin = x.size(); B, V, Fin = int(B), int(V), int(Fin) 

        # 重缩放拉普拉斯矩阵
        lmax = lmax_L(L)
        L = rescale_L(L, lmax) 
        
        # 把 scipy 稀疏矩阵 L 转成 pytorch
        L = L.tocoo()
        indices = np.column_stack((L.row, L.col)).T 
        indices = indices.astype(np.int64)
        indices = torch.from_numpy(indices)
        indices = indices.type(torch.LongTensor)
        L_data = L.data.astype(np.float32)
        L_data = torch.from_numpy(L_data) 
        L_data = L_data.type(torch.FloatTensor)
        L = torch.sparse.FloatTensor(indices, L_data, torch.Size(L.shape))
        L.requires_grad_(False)
        if torch.cuda.is_available():
            L = L.cuda()
        
        # 转换到 Chebyshev 基
        # 
        # 你的代码
        # 输入
        # x B x V x Fin
        # cl 线性层 Fin*K x Fout
        # L 拉普拉斯矩阵  lmax 最大特征值
        # 输出应该是 B x V x Fout
        #
        
        return x
        
        
    # 大小为 p 的最大池化。p 必须是 2 的幂。
    def graph_max_pool(self, x, p): 
        # 
        # 你的代码
        # 输入 B x V x F 输出 B x V/p x F
        #
        return x    
        
        
    def forward(self, x, d, L, lmax):
        # 图 CL1
        x = x.unsqueeze(2)  # B x V x Fin=1
        x = self.graph_conv_cheby(x, self.cl1, L[0], lmax[0], self.CL1_F, self.CL1_K)
        x = F.relu(x)
        x = self.graph_max_pool(x, 4)
        # 图 CL2
        x = self.graph_conv_cheby(x, self.cl2, L[2], lmax[2], self.CL2_F, self.CL2_K)
        x = F.relu(x)
        x = self.graph_max_pool(x, 4)
        # FC1
        x = x.view(-1, self.FC1Fin)
        x = self.fc1(x)
        x = F.relu(x)
        x  = nn.Dropout(d)(x)
        # FC2
        x = self.fc2(x)
        return x
        
        
    def loss(self, y, y_target, l2_regularization):
    
        loss = nn.CrossEntropyLoss()(y,y_target)

        l2_loss = 0.0
        for param in self.parameters():
            data = param* param
            l2_loss += data.sum()
           
        loss += 0.5* l2_regularization* l2_loss
            
        return loss
    
    
    def update(self, lr):
                
        update = torch.optim.SGD( self.parameters(), lr=lr, momentum=0.9 )
        
        return update
        
        
    def update_learning_rate(self, optimizer, lr):
   
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        return optimizer

    
    def evaluation(self, y_predicted, test_l):
    
        _, class_predicted = torch.max(y_predicted.data, 1)
        return 100.0* (class_predicted == test_l).sum()/ y_predicted.size(0)

In [ ]:
# 如果网络已存在则删除
try:
    del net
    print('Delete existing network\n')
except NameError:
    print('No existing network to delete\n')

# 网络参数
D = train_data.shape[1]
CL1_F = 32
CL1_K = 25
CL2_F = 64
CL2_K = 25
FC1_F = 512
FC2_F = 10
net_parameters = [D, CL1_F, CL1_K, CL2_F, CL2_K, FC1_F, FC2_F]
dropout_value = 0.5

# 实例化该类的对象 net
net = Graph_ConvNet_LeNet5(net_parameters)
if torch.cuda.is_available():
    net.cuda()
print(net)

好时机，检查一下你的网络是否正常工作……


In [ ]:
train_x, train_y = train_data[:5,:], train_labels[:5]
train_x =  torch.FloatTensor(train_x).type(dtypeFloat)
train_y = train_y.astype(np.int64)
train_y = torch.LongTensor(train_y).type(dtypeLong) 
            
# 前向传播
y = net(train_x, dropout_value, L, lmax)
print(y.shape)

In [ ]:
# 权重
L_net = list(net.parameters())

# 学习参数
learning_rate = 0.05
l2_regularization = 5e-4 
batch_size = 100
num_epochs = 3
train_size = train_data.shape[0]
nb_iter = int(num_epochs * train_size) // batch_size
print('num_epochs=',num_epochs,', train_size=',train_size,', nb_iter=',nb_iter)

# 优化器
global_lr = learning_rate
global_step = 0
decay = 0.95
decay_steps = train_size
lr = learning_rate
optimizer = net.update(lr) 

# 按 epoch 循环
indices = collections.deque()
for epoch in range(num_epochs):  # 多次遍历数据集

    # 重新打乱
    indices.extend(np.random.permutation(train_size))  # 随机排列
    
    # 重置时间
    t_start = time.time()
    
    # 提取 batches
    running_loss = 0.0
    running_accuray = 0
    running_total = 0
    while len(indices) >= batch_size:
        
        # 提取 batches
        batch_idx = [indices.popleft() for i in range(batch_size)]
        train_x, train_y = train_data[batch_idx,:], train_labels[batch_idx]
        train_x =  torch.FloatTensor(train_x).type(dtypeFloat)
        train_y = train_y.astype(np.int64)
        train_y = torch.LongTensor(train_y).type(dtypeLong) 
            
        # 前向传播
        y = net(train_x, dropout_value, L, lmax)
        loss = net.loss(y,train_y,l2_regularization) 
        loss_train = loss.detach().item()
        # 准确率
        acc_train = net.evaluation(y,train_y.data)
        # 反向传播
        loss.backward()
        # 更新
        global_step += batch_size  # 用于更新学习率
        optimizer.step()
        optimizer.zero_grad()
        # 损失、准确率
        running_loss += loss_train
        running_accuray += acc_train
        running_total += 1
        # 打印
        if not running_total%100:  # 每 100 个小批次打印一次
            print('epoch= %d, i= %4d, loss(batch)= %.4f, accuray(batch)= %.2f' % (epoch+1, running_total, loss_train, acc_train))
          
    # 打印
    t_stop = time.time() - t_start
    print('epoch= %d, loss(train)= %.3f, accuracy(train)= %.3f, time= %.3f, lr= %.5f' % 
          (epoch+1, running_loss/running_total, running_accuray/running_total, t_stop, lr))
 
    # 更新学习率
    lr = global_lr * pow( decay , float(global_step// decay_steps) )
    optimizer = net.update_learning_rate(optimizer, lr)
    
    
    # 测试集
    with torch.no_grad():
        running_accuray_test = 0
        running_total_test = 0
        indices_test = collections.deque()
        indices_test.extend(range(test_data.shape[0]))
        t_start_test = time.time()
        while len(indices_test) >= batch_size:
            batch_idx_test = [indices_test.popleft() for i in range(batch_size)]
            test_x, test_y = test_data[batch_idx_test,:], test_labels[batch_idx_test]
            test_x = torch.FloatTensor(test_x).type(dtypeFloat)
            y = net(test_x, 0.0, L, lmax) 
            test_y = test_y.astype(np.int64)
            test_y = torch.LongTensor(test_y).type(dtypeLong)
            acc_test = net.evaluation(y,test_y.data)
            running_accuray_test += acc_test
            running_total_test += 1
        t_stop_test = time.time() - t_start_test
        print('  accuracy(test) = %.3f %%, time= %.3f' % (running_accuray_test / running_total_test, t_stop_test))

<font color='red'>问题：通过修改拉普拉斯矩阵，有没有可能去掉重缩放这一步？</font>
